In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Import claims_data.csv and cust_data.csv which is provided to you and combine the two datasets appropriately to create a 360-degree view of the data. Use the same for the subsequent questions.

In [43]:
# Load the datasets
claims_df = pd.read_csv('claims.csv')
cust_df = pd.read_csv('cust_demographics.csv')

In [44]:
# Shape Check 
print('Claims shape: ', claims_df.shape)
print('Customer shape: ', cust_df.shape)

Claims shape:  (1100, 10)
Customer shape:  (1085, 6)


In [45]:
# Head of the data
print(claims_df.head(4))
print(cust_df.head(4))

   claim_id  customer_id      incident_cause  claim_date claim_area  \
0  54004764     21868593        Driver error  11/27/2017       Auto   
1  33985796     75740424               Crime  10/03/2018       Home   
2  53522022     30308357  Other driver error  02/02/2018       Auto   
3  13015401     47830476      Natural causes  06/17/2018       Auto   

  police_report     claim_type claim_amount  total_policy_claims fraudulent  
0            No  Material only        $2980                  1.0         No  
1       Unknown  Material only        $2980                  3.0         No  
2            No  Material only      $3369.5                  1.0        Yes  
3            No  Material only        $1680                  1.0         No  
    CUST_ID  gender DateOfBirth State       Contact   Segment
0  21868593  Female   12-Jan-79    VT  789-916-8172  Platinum
1  75740424  Female   13-Jan-70    ME  265-543-1264    Silver
2  30308357  Female   11-Mar-84    TN  798-631-4758    Silver
3  478

In [46]:
# Merge: Inner Join on Customer ID
df = pd.merge(
    left = claims_df, 
    right = cust_df,
    left_on = 'customer_id',
    right_on = 'CUST_ID',
    how = 'inner'
)

In [47]:
# Drop the duplicate key column
df.drop(columns=['CUST_ID'], inplace=True)

In [48]:
# Verify the result
print(f'\nMerged Dataframe Shape: {df.shape}')
print(f'\nColumns after merge: \n{df.columns.tolist()}')
print(f'\nFirst 3 rows: \n{df.head(3)}')


Merged Dataframe Shape: (1085, 15)

Columns after merge: 
['claim_id', 'customer_id', 'incident_cause', 'claim_date', 'claim_area', 'police_report', 'claim_type', 'claim_amount', 'total_policy_claims', 'fraudulent', 'gender', 'DateOfBirth', 'State', 'Contact', 'Segment']

First 3 rows: 
   claim_id  customer_id      incident_cause  claim_date claim_area  \
0  54004764     21868593        Driver error  11/27/2017       Auto   
1  33985796     75740424               Crime  10/03/2018       Home   
2  53522022     30308357  Other driver error  02/02/2018       Auto   

  police_report     claim_type claim_amount  total_policy_claims fraudulent  \
0            No  Material only        $2980                  1.0         No   
1       Unknown  Material only        $2980                  3.0         No   
2            No  Material only      $3369.5                  1.0        Yes   

   gender DateOfBirth State       Contact   Segment  
0  Female   12-Jan-79    VT  789-916-8172  Platinum  
1

### 2. Perform a data audit for the data types and find out if there are any mismatch within the current data types of the columns and their business significance.

In [49]:
# Step 1: See what pandas currently thinks each columns is
print(df.dtypes)

claim_id                 int64
customer_id              int64
incident_cause          object
claim_date              object
claim_area              object
police_report           object
claim_type              object
claim_amount            object
total_policy_claims    float64
fraudulent              object
gender                  object
DateOfBirth             object
State                   object
Contact                 object
Segment                 object
dtype: object


In [ ]:
# Step 2: Build a formal audit table
# ---> For each column - document: current type, expected type, mismatch, reason


audit_data = {
    'Column': ['claim_id', 'customer_id', 'incident_cause', 'claim_date',
                'claim_area', 'police_report', 'claim_type', 'claim_amount',
                'total_policy_claims', 'fraudulent','gender', 'DateOfBirth', 
                'State', 'Contact', 'Segment'],
    
    'Current Type': ['int64', 'int64', 'object', 'object',
                    'object', 'object', 'object', 'object',
                    'float64', 'object',
                    'object', 'object', 'object', 'object', 'object'],
    
    'Expected Type': ['int64', 'int64', 'object', 'datetime64',
                      'object', 'object', 'object', 'float64',
                      'int64', 'object',
                      'object', 'datetime64', 'object', 'object', 'object'],
    
    'Mismatch'     : ['No', 'No', 'No', 'YES',
                      'No', 'No', 'No', 'YES',
                      'YES', 'No', 'No', 'YES', 
                      'No', 'No', 'No'],
    
    'Business Reason': [
        'ID — no math needed',
        'ID — no math needed',
        'Category label — text is fine',
        'Date of claim — needs datetime for filtering & month extraction',
        'Category label — text is fine',
        'Category label — text is fine',
        'Category label — text is fine',
        'Money amount — has $ sign, must convert to float for calculations',
        'Count of claims — should be whole number, not 1.0',
        'Yes/No flag — text is fine',
        'Category label — text is fine',
        'Date of birth — needs datetime to calculate age',
        'State code — text is fine',
        'Phone number — not used in analysis',
        'Customer tier — text is fine'
    ]
}

audit_df = pd.DataFrame(audit_data)
print(audit_df.to_string(index=False))

             Column Current Type Expected Type Mismatch                                                   Business Reason
           claim_id        int64         int64       No                                               ID — no math needed
        customer_id        int64         int64       No                                               ID — no math needed
     incident_cause       object        object       No                                     Category label — text is fine
         claim_date       object    datetime64      YES   Date of claim — needs datetime for filtering & month extraction
         claim_area       object        object       No                                     Category label — text is fine
      police_report       object        object       No                                     Category label — text is fine
         claim_type       object        object       No                                     Category label — text is fine
       claim_amount     

In [63]:
# Step 3: Summary of mismatches
mismatches = audit_df[audit_df['Mismatch'] == 'YES']
print(f'\nTotal Mismatches Found: {len(mismatches)}')

print('\nColumns needing fix: ')

for _, row in mismatches.iterrows(): 
    print(f'-> {row['Column']}: "{row['Current Type']}" should be "{row['Expected Type']}"')  


Total Mismatches Found: 4

Columns needing fix: 
-> claim_date: "object" should be "datetime64"
-> claim_amount: "object" should be "float64"
-> total_policy_claims: "float64" should be "int64"
-> DateOfBirth: "object" should be "datetime64"


### 3. Convert the column claim_amount to numeric. Use the appropriate modules/attributes to remove the $ sign.

In [69]:
# Before Conversion: check current state
print('BEFORE: ')
print(f'dtype: {df['claim_amount'].dtype}')
print(f'sample value: {df['claim_amount'].head().tolist()}')
print(f'missing values: {df['claim_amount'].isnull().sum()}')

BEFORE: 
dtype: object
sample value: ['$2980', '$2980', '$3369.5', '$1680', '$2680']
missing values: 65


In [70]:
# Step 1: Remove the '$' character using string replace
df['claim_amount'] = df['claim_amount'].str.replace('$', '', regex=False)

In [ ]:
# Step 2: Convert the now-clean text column to numeric (float)
df['claim_amount'] = pd.to_numeric(df['claim_amount'], errors='coerce')

In [76]:
# After: verify the fix
print('\nAFTER:')
print(f'dtype: {df['claim_amount'].dtype}')
print(f'sample values: {df['claim_amount'].head().tolist()}')
print(f'missing values: {df['claim_amount'].isnull().sum()}')
print(f'min_value: {df['claim_amount'].min()}')
print(f'max_value: {df['claim_amount'].max()}')
print(f'mean_value: {df['claim_amount'].mean():.2f}')


AFTER:
dtype: float64
sample values: [2980.0, 2980.0, 3369.5, 1680.0, 2680.0]
missing values: 65
min_value: 1000.0
max_value: 48150.5
mean_value: 12467.68
